# Lab: Multi-Document RAG System

### 1. Environment Setup
**Python Version:** 3.10+

We install the core stack: `unstructured` for partitioning documents, `chromadb` as our vector database, `groq` for Llama 3.1 inference, and `sentence-transformers` for local embedding generation.

In [10]:
# 1. Install system dependencies
!apt-get install -y libmagic-dev poppler-utils tesseract-ocr --quiet

# 2. Install Python packages individually to isolate build errors
!pip install -q unstructured chromadb groq pandas pypdf python-dotenv sentence-transformers langchain-text-splitters

import importlib.util
import sys

# Verification script
packages = ['unstructured', 'chromadb', 'groq', 'sentence_transformers']
for pkg in packages:
    if importlib.util.find_spec(pkg) is None:
        print(f'❌ {pkg} NOT found. Please re-run this cell.')
    else:
        print(f'✅ {pkg} is installed.')

print("\nNOTE: If you still see ModuleNotFoundError in the next cell, click 'Runtime' -> 'Restart session' to refresh the environment.")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
libmagic-dev is already the newest version (1:5.41-3ubuntu0.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
✅ unstructured is installed.
✅ chromadb is installed.
✅ groq is installed.
✅ sentence_transformers is installed.

NOTE: If you still see ModuleNotFoundError in the next cell, click 'Runtime' -> 'Restart session' to refresh the environment.


### 2. Multi-Document Architecture

Our pipeline follows a **Normalize -> Embed -> Retrieve -> Synthesize** flow:

1.  **Ingestion & Parsing:** We use `Unstructured.io` to partition files. It identifies 'Elements' (Title, NarrativeText, Table) allowing us to treat a PDF or a CSV through a unified interface.
2.  **Metadata Tagging:** Every chunk is tagged with `source`, `file_type`, and `page_number` (if applicable) to ensure source attribution.
3.  **Vector Store:** `ChromaDB` stores the embeddings generated by `all-MiniLM-L6-v2`.
4.  **Retrieval:** A similarity search is performed. We can apply boolean filters on metadata (e.g., search only PDFs).
5.  **Synthesis:** Groq Llama 3.1 processes the context and query to generate an answer with citations.

In [12]:
import os
import pandas as pd
from typing import List, Dict
from groq import Groq
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from unstructured.partition.auto import partition
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuration
GROQ_API_KEY = "YOUR_GROQ_API_KEY" # Use Colab Secrets or .env
client = Groq(api_key=GROQ_API_KEY)

# Initialize Embedding Model
model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(model_name)

# Initialize ChromaDB
chroma_client = chromadb.Client()
# Use get_or_create_collection to prevent errors if the collection already exists
collection = chroma_client.get_or_create_collection(name="multi_doc_rag")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 3. Unified Ingestion Pipeline

This function handles PDFs, CSVs, and TXT files using a modular strategy.

In [13]:
def process_document(file_path: str) -> List[Dict]:
    """Extracts content and adds metadata using Unstructured.io."""
    file_name = os.path.basename(file_path)
    file_ext = os.path.splitext(file_name)[1].lower()

    # Use Unstructured for automatic partitioning
    elements = partition(filename=file_path)

    chunks = []
    for i, el in enumerate(elements):
        text = str(el)
        if len(text) < 20: continue # Skip noise

        chunks.append({
            "content": text,
            "metadata": {
                "source": file_name,
                "type": file_ext,
                "element_id": i
            }
        })
    return chunks

def ingest_folder(folder_path: str):
    all_docs = []
    for f in os.listdir(folder_path):
        path = os.path.join(folder_path, f)
        if os.path.isfile(path):
            print(f"Processing: {f}")
            all_docs.extend(process_document(path))

    # Add to ChromaDB
    for doc in all_docs:
        collection.add(
            documents=[doc['content']],
            metadatas=[doc['metadata']],
            ids=[f"{doc['metadata']['source']}_{doc['metadata']['element_id']}"]
        )
    print(f"Ingested {len(all_docs)} segments into ChromaDB.")

In [14]:
def query_rag(user_query: str, doc_filter: str = None):
    # 1. Retrieve
    where_clause = {"type": doc_filter} if doc_filter else None
    results = collection.query(
        query_texts=[user_query],
        n_results=5,
        where=where_clause
    )

    context = "\n".join(results['documents'][0])
    sources = [m['source'] for m in results['metadatas'][0]]

    # 2. Synthesize with Groq
    prompt = f"""Answer the question using ONLY the context provided.
    Context: {context}
    Question: {user_query}
    Instructions: Cite your sources. If information is conflicting, mention both sources."""

    completion = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "answer": completion.choices[0].message.content,
        "sources": list(set(sources))
    }

### 4. Testing & Evaluation

Run these queries once you have files uploaded to a directory (e.g., `./data/`).

1.  **Cross-Doc Summary:** "Summarize the policies mentioned across all documents."
2.  **Specific Data:** "What sales data appears in the CSV?"
3.  **Comparison:** "Compare information from the PDF and the text files."
4.  **Constraint Check:** "What does the PDF say about security?" (Use filter)
5.  **Hallucination Test:** "What is the capital of Mars?" (Should say 'not in context')"